# LeetCode 1941 — Check if All Characters Have Equal Number of Occurrences bằng Python

**Link đề:** https://leetcode.com/problems/check-if-all-characters-have-equal-number-of-occurrences/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Counting

```python
class Solution:
    def areOccurrencesEqual(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu bài toán **đếm tần suất** (frequency counting) — kỹ thuật nền của rất nhiều bài Hash Table.
2. Tự xây bảng đếm bằng `dict` (`d.get(k, 0) + 1`), rồi dùng `collections.Counter` và mảng 26 phần tử.
3. Biết kiểm tra "mọi giá trị bằng nhau" bằng `set(values)`, `all(...)`, hoặc so với một giá trị mốc.
4. So sánh baseline `str.count` lặp lại với cách đếm một lượt; trace từng bước, chứng minh đúng, phân tích Time/Space.
5. Tự giải các biến thể (LeetCode 1207, 2423…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — chuỗi "tốt"

Cho chuỗi `s`. `s` gọi là **good** nếu **mọi ký tự xuất hiện trong `s`** đều có **cùng số lần xuất hiện** (cùng tần suất). Trả `True` nếu `s` good, ngược lại `False`.

| `s` | Output | Giải thích |
|---|:---:|---|
| `"abacbc"` | `True` | `a`, `b`, `c` đều xuất hiện 2 lần |
| `"aaabb"` | `False` | `a` 3 lần, `b` 2 lần |

**Constraints:** `1 <= s.length <= 1000`, `s` chỉ gồm chữ thường tiếng Anh.

Chú ý cụm "ký tự **xuất hiện** trong `s`": chữ không xuất hiện (tần suất 0) **không** được tính. Nếu tính cả chúng, gần như mọi chuỗi đều "không tốt".

In [1]:
# Đếm bằng mắt cho ví dụ 1
s = "abacbc"
for ch in "abc":
    print(ch, "xuất hiện", s.count(ch), "lần")

a xuất hiện 2 lần
b xuất hiện 2 lần
c xuất hiện 2 lần


## 2. Nền tảng Python cần dùng

### 2.1. `str.count(x)` — đếm số lần `x` xuất hiện

`s.count("a")` duyệt **toàn bộ** chuỗi để đếm → tốn `O(n)` mỗi lần gọi. Dễ dùng nhưng nếu gọi nhiều lần sẽ lặp lại công việc.

In [2]:
s = "aaabb"
print(s.count("a"), s.count("b"), s.count("z"))   # 3 2 0

3 2 0


### 2.2. `dict` làm bảng đếm và `dict.get(key, default)`

`dict` lưu cặp **key → value**. Ở đây key là ký tự, value là số lần xuất hiện.

- `d[k]` đọc giá trị; nếu `k` chưa có → lỗi `KeyError`.
- `d.get(k, 0)` đọc giá trị, nếu chưa có thì trả `0` (không lỗi). Đây là mẹo chuẩn để đếm: `d[ch] = d.get(ch, 0) + 1`.

In [3]:
freq = {}
for ch in "abacbc":
    freq[ch] = freq.get(ch, 0) + 1
    print(f"đọc {ch!r} -> freq = {freq}")
print(freq.keys(), freq.values())

đọc 'a' -> freq = {'a': 1}
đọc 'b' -> freq = {'a': 1, 'b': 1}
đọc 'a' -> freq = {'a': 2, 'b': 1}
đọc 'c' -> freq = {'a': 2, 'b': 1, 'c': 1}
đọc 'b' -> freq = {'a': 2, 'b': 2, 'c': 1}
đọc 'c' -> freq = {'a': 2, 'b': 2, 'c': 2}
dict_keys(['a', 'b', 'c']) dict_values([2, 2, 2])


### 2.3. `collections.Counter` — bảng đếm có sẵn

`Counter(s)` làm đúng việc của vòng lặp trên trong một dòng. Nó là một `dict` đặc biệt: truy cập key không có trả `0` thay vì lỗi.

In [4]:
from collections import Counter

c = Counter("aaabb")
print(c)              # Counter({'a': 3, 'b': 2})
print(c["a"], c["z"]) # 3 0  — key không có trả 0
print(list(c.values()))

Counter({'a': 3, 'b': 2})
3 0
[3, 2]


### 2.4. `set` để kiểm tra "mọi giá trị bằng nhau"

`set(dãy)` bỏ phần tử trùng. Mọi giá trị trong dãy bằng nhau ⇔ `len(set(dãy)) == 1` (với dãy không rỗng).

In [5]:
print(set([2, 2, 2]), len(set([2, 2, 2])))   # {2} 1
print(set([3, 2]), len(set([3, 2])))         # {2, 3} 2

{2} 1
{2, 3} 2


### 2.5. `ord` / `chr` và mảng 26 phần tử

Với chữ thường, `ord(ch) - ord('a')` cho chỉ số `0..25`. Một list 26 số `[0] * 26` là bảng đếm cố định kích thước — nhanh, không cần băm.

In [6]:
print(ord("a"), ord("c"), ord("c") - ord("a"))   # 97 99 2
cnt = [0] * 26
for ch in "abacbc":
    cnt[ord(ch) - ord("a")] += 1
print(cnt[:5])                                   # [2, 2, 2, 0, 0]
print(chr(ord("a") + 2))                         # 'c'

97 99 2
[2, 2, 2, 0, 0]
c


## 3. Tính tay — lập bảng tần suất

`s = "aaabb"`:

| Ký tự | Đếm |
|---|:---:|
| a | 3 |
| b | 2 |

Hai giá trị `3 ≠ 2` → `False`.

`s = "abacbc"`: `a:2, b:2, c:2` → một giá trị duy nhất `{2}` → `True`.

Như vậy bài chia thành **hai bước**: (1) dựng bảng tần suất; (2) kiểm tra các tần suất **khác 0** có bằng nhau không.

## 4. Cách 1 — Baseline: gọi `s.count` cho từng ký tự

Lấy tần suất của ký tự đầu `s[0]` làm **mốc**. Duyệt từng ký tự `ch` của `s`, nếu `s.count(ch) != mốc` → `False`.

In [7]:
def are_equal_count(s):
    target = s.count(s[0])
    for ch in s:
        if s.count(ch) != target:
            return False
    return True

print(are_equal_count("abacbc"))   # True
print(are_equal_count("aaabb"))    # False

True
False


### 4.1. Nút thắt của baseline

Mỗi lần `s.count(ch)` duyệt cả chuỗi `O(n)`, và ta gọi nó `n` lần → `O(n²)`. Với `n = 1000` là khoảng 10⁶ phép, vẫn qua được, nhưng ta **đếm lại cùng một ký tự nhiều lần** (chuỗi `"aaaa..."` đếm `a` tới `n` lần).

Cải tiến nhỏ: chỉ gọi `count` cho mỗi ký tự **khác nhau** (`set(s)`, tối đa 26) → `O(26·n)`. Cải tiến thật sự: **đếm tất cả trong một lượt**.

In [8]:
def are_equal_count_distinct(s):
    counts = [s.count(ch) for ch in set(s)]   # tối đa 26 lần gọi count
    return len(set(counts)) == 1

print(are_equal_count_distinct("abacbc"), are_equal_count_distinct("aaabb"))

# Đếm số lần "chạm" ký tự của mỗi cách trên chuỗi 1000 chữ 'a'
s_big = "a" * 1000
print("baseline:", len(s_big) * len(s_big), "lần chạm | distinct:", len(set(s_big)) * len(s_big), "lần chạm | một lượt:", len(s_big))

True False
baseline: 1000000 lần chạm | distinct: 1000 lần chạm | một lượt: 1000


## 5. Bước cải tiến — đếm một lượt rồi so sánh các tần suất

Insight: **không cần biết vị trí**, chỉ cần biết *mỗi ký tự xuất hiện bao nhiêu lần*. Một lượt duyệt chuỗi là đủ để dựng toàn bộ bảng tần suất (`O(n)`), sau đó số giá trị cần so ≤ 26.

## 6. Lời giải chính — `dict` tự đếm, giảng từng đoạn

**Đoạn 1 — dựng bảng:** `freq[ch] = freq.get(ch, 0) + 1` cho mỗi `ch`.

**Đoạn 2 — lấy mốc:** `target` = một tần suất bất kỳ, ví dụ tần suất của `s[0]`.

**Đoạn 3 — so sánh:** mọi `v` trong `freq.values()` phải bằng `target`.

In [9]:
# Đoạn 1 riêng lẻ
s = "aaabb"
freq = {}
for ch in s:
    freq[ch] = freq.get(ch, 0) + 1
print("freq =", freq)
# Đoạn 2 + 3
target = freq[s[0]]
print("target =", target, "| các giá trị:", list(freq.values()))
print(all(v == target for v in freq.values()))

freq = {'a': 3, 'b': 2}
target = 3 | các giá trị: [3, 2]
False


In [10]:
def are_equal_dict(s):
    freq = {}
    for ch in s:                              # đoạn 1
        freq[ch] = freq.get(ch, 0) + 1
    target = freq[s[0]]                       # đoạn 2
    for v in freq.values():                   # đoạn 3
        if v != target:
            return False
    return True

for t in ["abacbc", "aaabb", "a", "zzz", "abcabcab"]:
    print(repr(t), are_equal_dict(t))

'abacbc' True
'aaabb' False
'a' True
'zzz' True
'abcabcab' False


## 7. TRACE từng bước

Ví dụ `s = "abacbc"`.

**Pha 1 — đếm:**

| i | ch | freq trước | freq sau |
|---:|:---:|---|---|
| 0 | a | `{}` | `{a:1}` |
| 1 | b | `{a:1}` | `{a:1, b:1}` |
| 2 | a | `{a:1, b:1}` | `{a:2, b:1}` |
| 3 | c | `{a:2, b:1}` | `{a:2, b:1, c:1}` |
| 4 | b | … | `{a:2, b:2, c:1}` |
| 5 | c | … | `{a:2, b:2, c:2}` |

**Pha 2 — so sánh:** `target = freq['a'] = 2`; lần lượt `2 == 2`, `2 == 2`, `2 == 2` → `True`.

In [11]:
def trace_are_equal(s):
    print(f"s = {s!r}")
    freq = {}
    for i, ch in enumerate(s):
        before = dict(freq)            # chụp lại trạng thái TRƯỚC khi cập nhật
        freq[ch] = freq.get(ch, 0) + 1
        print(f"  i={i} ch={ch!r} freq: {before} -> {freq}")
    target = freq[s[0]]
    print(f"  target = freq[{s[0]!r}] = {target}")
    for ch, v in freq.items():
        print(f"  so sánh {ch!r}: {v} == {target}? {v == target}")
        if v != target:
            print("  -> khác mốc, trả False")
            return False
    print("  -> mọi tần suất bằng nhau, trả True")
    return True

trace_are_equal("abacbc")
print()
trace_are_equal("aaabb")

s = 'abacbc'
  i=0 ch='a' freq: {} -> {'a': 1}
  i=1 ch='b' freq: {'a': 1} -> {'a': 1, 'b': 1}
  i=2 ch='a' freq: {'a': 1, 'b': 1} -> {'a': 2, 'b': 1}
  i=3 ch='c' freq: {'a': 2, 'b': 1} -> {'a': 2, 'b': 1, 'c': 1}
  i=4 ch='b' freq: {'a': 2, 'b': 1, 'c': 1} -> {'a': 2, 'b': 2, 'c': 1}
  i=5 ch='c' freq: {'a': 2, 'b': 2, 'c': 1} -> {'a': 2, 'b': 2, 'c': 2}
  target = freq['a'] = 2
  so sánh 'a': 2 == 2? True
  so sánh 'b': 2 == 2? True
  so sánh 'c': 2 == 2? True
  -> mọi tần suất bằng nhau, trả True

s = 'aaabb'
  i=0 ch='a' freq: {} -> {'a': 1}
  i=1 ch='a' freq: {'a': 1} -> {'a': 2}
  i=2 ch='a' freq: {'a': 2} -> {'a': 3}
  i=3 ch='b' freq: {'a': 3} -> {'a': 3, 'b': 1}
  i=4 ch='b' freq: {'a': 3, 'b': 1} -> {'a': 3, 'b': 2}
  target = freq['a'] = 3
  so sánh 'a': 3 == 3? True
  so sánh 'b': 2 == 3? False
  -> khác mốc, trả False


False

### 7.1. `enumerate` và `dict.items()`

- `enumerate(s)` cho cặp `(chỉ số, phần tử)` — tiện khi vừa cần `i` vừa cần `ch`.
- `freq.items()` cho cặp `(key, value)`.
- `dict(freq)` tạo **bản sao**; nếu chỉ viết `before = freq` thì `before` và `freq` là **cùng một** dict, in ra sẽ thấy giống nhau.

In [12]:
for i, ch in enumerate("xyz"):
    print(i, ch)
d = {"a": 1}
alias = d          # không sao chép
copy = dict(d)     # sao chép
d["a"] = 99
print(alias, copy)  # {'a': 99} {'a': 1}

0 x
1 y
2 z
{'a': 99} {'a': 1}


## 8. Các cách giải khác

### 8.1. Cách 3 — `Counter` + `set` (một dòng)

In [13]:
def are_equal_counter(s):
    return len(set(Counter(s).values())) == 1

print(are_equal_counter("abacbc"), are_equal_counter("aaabb"))

True False


### 8.2. Cách 4 — mảng 26 phần tử, **bỏ qua số 0**

Mảng đếm có 26 ô, các chữ không xuất hiện có giá trị 0 — phải loại chúng trước khi so sánh (xem Lỗi 2 ở mục 13).

In [14]:
def are_equal_array(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - ord("a")] += 1
    nonzero = [c for c in cnt if c > 0]
    return len(set(nonzero)) == 1

print(are_equal_array("abacbc"), are_equal_array("aaabb"))

True False


### 8.3. Cách 5 — góc nhìn toán học: `n = k · số_ký_tự_khác_nhau`

Nếu có `d` ký tự khác nhau và mỗi ký tự xuất hiện đúng `k` lần thì `n = k·d`. Vậy điều kiện tương đương: mọi tần suất bằng `n // d` (và `n % d == 0`). Không cần lấy mốc từ `s[0]`.

In [15]:
def are_equal_math(s):
    freq = Counter(s)
    d = len(freq)
    if len(s) % d != 0:            # dừng sớm: chia không hết thì chắc chắn không đều
        return False
    k = len(s) // d
    return all(v == k for v in freq.values())

for t in ["abacbc", "aaabb", "abcabcab"]:
    print(repr(t), "n =", len(t), "d =", len(set(t)), "->", are_equal_math(t))

'abacbc' n = 6 d = 3 -> True
'aaabb' n = 5 d = 2 -> False
'abcabcab' n = 8 d = 3 -> False


### 8.4. So sánh `min` và `max`

Mọi giá trị bằng nhau ⇔ giá trị nhỏ nhất bằng giá trị lớn nhất.

In [16]:
def are_equal_minmax(s):
    values = Counter(s).values()
    return min(values) == max(values)

print(are_equal_minmax("abacbc"), are_equal_minmax("aaabb"))

True False


## 9. Code nộp LeetCode (Python 3)

In [17]:
class Solution:
    def areOccurrencesEqual(self, s: str) -> bool:
        freq = {}
        for ch in s:
            freq[ch] = freq.get(ch, 0) + 1
        target = freq[s[0]]
        return all(v == target for v in freq.values())


sol = Solution()
assert sol.areOccurrencesEqual("abacbc") is True
assert sol.areOccurrencesEqual("aaabb") is False
print("2 ví dụ của đề PASS")

2 ví dụ của đề PASS


Bản ngắn với `Counter`:

```python
from collections import Counter

class Solution:
    def areOccurrencesEqual(self, s: str) -> bool:
        return len(set(Counter(s).values())) == 1
```

## 10. Vì sao thuật toán đúng?

- **Pha 1:** sau khi duyệt xong, `freq[c]` đúng bằng số lần `c` xuất hiện trong `s`. Invariant: *sau khi đọc `s[:i]`, `freq[c]` = số lần `c` xuất hiện trong `s[:i]`*. Bước `i` chỉ tăng đúng ô của `s[i]` thêm 1 → invariant giữ nguyên với `s[:i+1]`.
- `freq` chỉ chứa ký tự **thực sự xuất hiện** (ta chỉ thêm key khi đọc tới nó) → khớp định nghĩa "mọi ký tự xuất hiện trong `s`".
- **Pha 2:** "mọi tần suất bằng nhau" ⇔ "mọi tần suất bằng một tần suất cụ thể nào đó" (ví dụ của `s[0]`, chắc chắn tồn tại vì `len(s) >= 1`). Hàm trả `True` đúng khi không có giá trị nào khác mốc.

## 11. Time / Space Complexity

Gọi `n = len(s)`, `σ = 26` (kích thước bảng chữ).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. `s.count` cho từng ký tự | `O(n²)` | `O(1)` | baseline |
| 1b. `s.count` cho ký tự khác nhau | `O(σ·n)` | `O(σ)` | cải tiến nhỏ |
| 2. `dict` tự đếm (chính) | `O(n)` | `O(σ)` | nên nộp, dễ giải thích |
| 3. `Counter` + `set` | `O(n)` | `O(σ)` | ngắn nhất |
| 4. Mảng 26 | `O(n + σ)` | `O(σ)` | nhanh, nhớ bỏ số 0 |
| 5. Toán `n = k·d` | `O(n)` | `O(σ)` | góc nhìn hay |

Bộ nhớ `O(σ)` = `O(26)` = hằng số, nên cũng có thể viết `O(1)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.areOccurrencesEqual("abacbc"), True)
check("đề 2", sol.areOccurrencesEqual("aaabb"), False)
check("một ký tự", sol.areOccurrencesEqual("a"), True)
check("một loại ký tự lặp", sol.areOccurrencesEqual("zzzz"), True)
check("mỗi chữ 1 lần", sol.areOccurrencesEqual("abcdefghijklmnopqrstuvwxyz"), True)
check("lệch ở chữ cuối", sol.areOccurrencesEqual("aabbc"), False)
check("lệch ở chữ đầu", sol.areOccurrencesEqual("abbcc"), False)
check("n chia hết d nhưng không đều", sol.areOccurrencesEqual("aaaabbcccccc"), False)
check("thứ tự lộn xộn", sol.areOccurrencesEqual("cabbac"), True)

PASS đề 1: True
PASS đề 2: False
PASS một ký tự: True
PASS một loại ký tự lặp: True
PASS mỗi chữ 1 lần: True
PASS lệch ở chữ cuối: False
PASS lệch ở chữ đầu: False
PASS n chia hết d nhưng không đều: False
PASS thứ tự lộn xộn: True


### 12.1. Expected phải được **tính**, không được đoán

Khi tự nghĩ test, rất dễ "đoán" expected sai. Ví dụ chuỗi `"aaab" + "bbcc" + "c"` nhìn lộn xộn, dễ tưởng là `False` — nhưng ghép lại nó là `"aaabbbccc"`, mỗi chữ đúng 3 lần → **True**. Cách an toàn: in `Counter` ra trước khi ghi expected.

In [19]:
print(Counter("aaab" + "bbcc" + "c"))     # a:3 b:3 c:3 -> thật ra là chuỗi TỐT
t = "aaaabbcc"                            # n=8,  d=3: 8 % 3 != 0 -> chắc chắn không đều
w = "aaaabbcccccc"                        # n=12, d=3: 12 % 3 == 0 nhưng a:4 b:2 c:6
for x in [t, w]:
    print(repr(x), dict(Counter(x)), "n % d =", len(x) % len(set(x)), "->", sol.areOccurrencesEqual(x))

Counter({'a': 3, 'b': 3, 'c': 3})
'aaaabbcc' {'a': 4, 'b': 2, 'c': 2} n % d = 2 -> False
'aaaabbcccccc' {'a': 4, 'b': 2, 'c': 6} n % d = 0 -> False


Bài học kiểm thử: **expected phải được tính, không được đoán**. Chuỗi `w = "aaaabbcccccc"` mới là phản ví dụ đúng nghĩa cho ý "chỉ cần `n % d == 0` là đủ" — `12 % 3 == 0` nhưng tần suất `4, 2, 6` không đều. Vì vậy cách 5 phải kiểm tra **cả** `all(v == k ...)`, không chỉ phép chia hết.

In [20]:
# Đối chiếu tất cả các cách trên dữ liệu ngẫu nhiên có seed + vét cạn chuỗi ngắn trên "abc"
import random
from itertools import product

funcs = [are_equal_count, are_equal_count_distinct, are_equal_dict, are_equal_counter,
         are_equal_array, are_equal_math, are_equal_minmax, Solution().areOccurrencesEqual]
checked = 0
for length in range(1, 8):
    for tup in product("abc", repeat=length):
        t = "".join(tup)
        assert len({f(t) for f in funcs}) == 1, t
        checked += 1
random.seed(1941)
for _ in range(300):
    t = "".join(random.choice("abcde") for _ in range(random.randint(1, 30)))
    assert len({f(t) for f in funcs}) == 1, t
    checked += 1
print("Đã đối chiếu", checked, "chuỗi — mọi cách khớp nhau")

Đã đối chiếu 3579 chuỗi — mọi cách khớp nhau


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Dùng `d[ch] += 1` khi key chưa tồn tại

In [21]:
def buggy_keyerror(s):
    freq = {}
    for ch in s:
        freq[ch] += 1          # KeyError ở lần đầu gặp ch
    return freq

try:
    buggy_keyerror("ab")
except KeyError as e:
    print("KeyError:", e)

KeyError: 'a'


Sửa: `freq[ch] = freq.get(ch, 0) + 1`, hoặc dùng `Counter` / `defaultdict(int)`.

### Lỗi 2 — So sánh cả các ô 0 trong mảng 26

In [22]:
def buggy_array(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - ord("a")] += 1
    return len(set(cnt)) == 1          # SAI: tập {0, 2} có 2 phần tử

print("buggy:", buggy_array("abacbc"), "| đúng:", are_equal_array("abacbc"))
print("buggy với đủ 26 chữ:", buggy_array("abcdefghijklmnopqrstuvwxyz"))   # tình cờ đúng

buggy: False | đúng: True
buggy với đủ 26 chữ: True


Chữ không xuất hiện có tần suất 0, nhưng đề chỉ xét chữ **xuất hiện**. Bản lỗi chỉ đúng khi chuỗi chứa đủ 26 chữ.

### Lỗi 3 — Chỉ kiểm tra `len(s) % số_ký_tự_khác_nhau == 0`

Như đã thấy ở mục 12: `"aaaabbcccccc"` thoả `12 % 3 == 0` nhưng không đều. Chia hết là điều kiện **cần**, không **đủ**.

In [23]:
def buggy_divisible_only(s):
    return len(s) % len(set(s)) == 0

print("buggy:", buggy_divisible_only("aaaabbcccccc"), "| đúng:", are_equal_dict("aaaabbcccccc"))

buggy: True | đúng: False


### Lỗi 4 — Chỉ so sánh **hai ký tự liền kề** trong chuỗi

```python
for i in range(len(s) - 1):
    if s.count(s[i]) != s.count(s[i + 1]): ...
```
Cách này tình cờ đúng (vì so sánh bắc cầu qua toàn chuỗi), nhưng vẫn `O(n²)` và khó đọc. Hãy tách rõ hai pha: **đếm**, rồi **so sánh các tần suất**.

### Lỗi 5 — Nhầm `freq.keys()` với `freq.values()`

`set(freq.keys())` là tập các **ký tự** — luôn có độ dài = số ký tự khác nhau, không liên quan tới tần suất. Phải dùng `values()`.

In [24]:
freq = Counter("aaabb")
print(set(freq.keys()), set(freq.values()))

{'b', 'a'} {2, 3}


## 14. Mẹo nhận dạng pattern

- Đề hỏi về "**số lần xuất hiện**", "tần suất", "mỗi ký tự… bao nhiêu lần" → **dựng bảng đếm** (`dict`/`Counter`/mảng 26) trong một lượt.
- "Mọi giá trị bằng nhau" → `len(set(values)) == 1` hoặc `min == max`.
- Bảng chữ nhỏ cố định (26 chữ thường) → mảng 26 phần tử là lựa chọn nhanh, nhưng nhớ **lọc số 0** khi đề chỉ nói về phần tử xuất hiện.
- Họ bài đếm tần suất: 242 (Valid Anagram), 387 (First Unique Character), 1207 (Unique Number of Occurrences), 2423 (Remove Letter To Equalize Frequency), 1897 (Redistribute Characters to Make All Strings Equal).

## 15. Biến thể

### 15.1. LeetCode 1207 — Unique Number of Occurrences

Ngược lại với bài này: mọi tần suất phải **khác nhau**. Điều kiện: số tần suất khác nhau = số phần tử khác nhau.

In [25]:
def unique_occurrences(arr):
    freq = Counter(arr)
    return len(set(freq.values())) == len(freq)

print(unique_occurrences([1, 2, 2, 1, 1, 3]))   # True: 1->3, 2->2, 3->1
print(unique_occurrences([1, 2]))               # False: cả hai đều 1 lần

True
False


### 15.2. LeetCode 2423 — xoá đúng một ký tự để mọi tần suất bằng nhau

Vét cạn: thử xoá từng ký tự khác nhau (giảm tần suất của nó đi 1), kiểm tra các tần suất **dương** còn lại có bằng nhau không. Chỉ tối đa 26 lần thử.

In [26]:
def equal_frequency_after_one_removal(word):
    freq = Counter(word)
    for ch in list(freq):
        freq[ch] -= 1
        values = [v for v in freq.values() if v > 0]
        ok = len(set(values)) == 1          # đề 2423 cho word dài >= 2 nên luôn còn ít nhất 1 chữ
        freq[ch] += 1                       # hoàn tác
        if ok:
            return True
    return False

print(equal_frequency_after_one_removal("abcc"))   # True: xoá một 'c'
print(equal_frequency_after_one_removal("aazz"))   # False

True
False


### 15.3. Nhóm ký tự theo tần suất

Đôi khi ta muốn biết "tần suất `k` gồm những chữ nào". Đảo ngược bảng đếm: key là tần suất, value là list ký tự.

In [27]:
def group_by_frequency(s):
    groups = {}
    for ch, v in Counter(s).items():
        groups.setdefault(v, []).append(ch)    # setdefault: tạo list rỗng nếu key chưa có
    return groups

print(group_by_frequency("aaabbc"))   # {3: ['a'], 2: ['b'], 1: ['c']}

{3: ['a'], 2: ['b'], 1: ['c']}


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `Counter("banana")` in ra gì? `len(set(Counter("banana").values()))` bằng bao nhiêu?

**Bài 2 (tính tay):** lập bảng tần suất cho `s = "xyzzyx"` và `s = "xxyyz"`, kết luận good hay không.

**Bài 3 (điền chỗ trống):** hoàn thiện bảng đếm bằng `dict`.

In [28]:
def exercise_build_freq(s):
    freq = {}
    for ch in s:
        # TODO: tăng freq[ch] mà không gây KeyError
        pass
    return freq

# assert exercise_build_freq("aab") == {"a": 2, "b": 1}

**Bài 4 (tự viết lại):** viết `exercise_are_equal(s)` dùng **mảng 26 phần tử**, nhớ bỏ qua số 0.

In [29]:
def exercise_are_equal(s):
    # TODO
    pass

# assert exercise_are_equal("abacbc") is True
# assert exercise_are_equal("aaabb") is False

**Bài 5 (biến thể):** viết `exercise_unique_occurrences(arr)` (LeetCode 1207).

**Bài 6 (mini challenge):** viết `exercise_min_deletions_to_equal(s)` trả **số ký tự ít nhất cần xoá** để chuỗi còn lại là good (được phép xoá hết một loại chữ). Ví dụ `"aaabbc"` → xoá còn mỗi loại 1 chữ (`a`,`b`,`c`) tốn 3, hoặc chỉ giữ `aaa` tốn 3, hoặc giữ `aabb` tốn 2 → đáp án `2`.

In [30]:
def exercise_unique_occurrences(arr):
    # TODO
    pass

def exercise_min_deletions_to_equal(s):
    # TODO: thử mọi tần suất mục tiêu k
    pass

# assert exercise_unique_occurrences([1, 2, 2, 1, 1, 3]) is True
# assert exercise_unique_occurrences([1, 2]) is False
# assert exercise_min_deletions_to_equal("aaabbc") == 2
# assert exercise_min_deletions_to_equal("abc") == 0

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | đếm từng chữ của banana | b:1, a:3, n:2 | ba giá trị khác nhau |
| 2 | dựng bảng | `xyzzyx`: mỗi chữ 2 lần | `xxyyz`: z chỉ 1 lần |
| 3 | `get` với mặc định | `freq.get(ch, 0)` | `freq[ch] = freq.get(ch, 0) + 1` |
| 4 | chỉ số `ord(ch) - 97` | lọc `c > 0` | `len(set(c for c in cnt if c)) == 1` |
| 5 | so số tần suất khác nhau với số phần tử khác nhau | `Counter` | `len(set(f.values())) == len(f)` |
| 6 | chọn tần suất đích `k` từ 1 tới max | chữ có tần suất `< k` phải xoá hết, `>= k` xoá bớt còn `k` | chi phí = `sum(v if v < k else v - k)`, lấy min |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `Counter({'a': 3, 'n': 2, 'b': 1})`, có `3` giá trị khác nhau.

**Bài 2:** `"xyzzyx"` → `x:2, y:2, z:2` → good. `"xxyyz"` → `x:2, y:2, z:1` → không good.

In [31]:
print(Counter("banana"), len(set(Counter("banana").values())))
print(are_equal_dict("xyzzyx"), are_equal_dict("xxyyz"))


def answer_build_freq(s):
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    return freq


def answer_are_equal(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - ord("a")] += 1
    return len(set(c for c in cnt if c > 0)) == 1


def answer_unique_occurrences(arr):
    f = Counter(arr)
    return len(set(f.values())) == len(f)


def answer_min_deletions_to_equal(s):
    values = list(Counter(s).values())
    best = len(s)                               # xoá gần hết vẫn là một cận trên
    for k in range(1, max(values) + 1):
        cost = sum(v if v < k else v - k for v in values)
        best = min(best, cost)
    return best


assert answer_build_freq("aab") == {"a": 2, "b": 1}
assert answer_are_equal("abacbc") is True
assert answer_are_equal("aaabb") is False
assert answer_unique_occurrences([1, 2, 2, 1, 1, 3]) is True
assert answer_unique_occurrences([1, 2]) is False
assert answer_min_deletions_to_equal("aaabbc") == 2
assert answer_min_deletions_to_equal("abc") == 0
for length in range(1, 7):
    for tup in product("abc", repeat=length):
        t = "".join(tup)
        assert answer_are_equal(t) == are_equal_dict(t)
print("Đáp án tham khảo PASS toàn bộ test!")

Counter({'a': 3, 'n': 2, 'b': 1}) 3
True False
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def areOccurrencesEqual(self, s: str) -> bool:
        freq = {}
        for ch in s:
            freq[ch] = freq.get(ch, 0) + 1      # pha 1: đếm
        return len(set(freq.values())) == 1     # pha 2: mọi tần suất bằng nhau
```

**Câu thần chú:** *đếm một lượt → nhìn tập các tần suất → chỉ một giá trị là tốt.*

**Checklist trước khi submit:**
- [ ] Đếm bằng `get(ch, 0) + 1` / `Counter`, không `KeyError`?
- [ ] Nếu dùng mảng 26: đã **bỏ số 0**?
- [ ] Dùng `values()` chứ không phải `keys()`?
- [ ] Không chỉ dựa vào `n % d == 0`?
- [ ] Test chuỗi một ký tự (`"a"` → True) và chuỗi lệch ở cuối (`"aabbc"` → False)?